# MEIDEM — Tutorial

**MEIDEM** (Multi-grid Epic Interpolator for stellar limb DarkEning Models) provides a unified interface for interpolating stellar limb darkening (LD) coefficients from multiple published grids.

This notebook covers:
1. [Installation and quick start](#1-installation-and-quick-start)
2. [Solar reference example (IAU 2015)](#2-solar-reference-example-iau-2015)
3. [Choosing the right grid](#3-choosing-the-right-grid)
4. [Choosing microturbulence (xi)](#4-choosing-microturbulence-xi)
5. [Processing a table of stars](#5-processing-a-table-of-stars)

---

## 1. Installation and quick start

Install MEIDEM from PyPI — all grid tables are bundled, no manual downloads required:

```bash
pip install meidem
```

The main function is `get_ld_coefficients()`. At minimum, you need to provide the stellar parameters, a passband, and a grid:

In [2]:
import meidem

print(f"MEIDEM version: {meidem.__version__}")

ImportError: cannot import name '__version__' from 'meidem._version' (/home/icaromeidem/Documentos/meidem/meidem/_version.py)

In [ ]:
# Minimal example — Sun-like star with TESS
result = meidem.get_ld_coefficients(
    teff     = 5778,          # effective temperature (K)
    logg     = 4.44,          # surface gravity log g (cgs)
    feh      = 0.0,           # metallicity [Fe/H] (dex)
    passband = 'TESS',
    grid     = 'kostogryz2022',
    law      = 'power2',
)

print(f"Coefficients : {result['coefficients']}")
print(f"Law          : {result['law']}")
print(f"Reference    : {result['reference']}")
print(f"DOI          : {result['doi']}")

The function returns a dict with all relevant metadata. You can explore all available grids, laws, and passbands using the discovery functions:

In [ ]:
# List all available grids
meidem.available_grids()

In [ ]:
# Laws available for a specific grid
meidem.available_laws('claret2017')

In [ ]:
# Passbands available for a specific grid
meidem.available_passbands('claret2022')

---

## 2. Solar reference example (IAU 2015)

A useful sanity check is to compute LD coefficients for the Sun using the IAU 2015 nominal solar parameters and compare results across grids.

| Parameter | Value | Reference |
|-----------|-------|-----------|
| Teff | 5772 K | Prša et al. (2016) |
| log g | 4.438 | GM☉/R☉² in cgs |
| [Fe/H] | 0.0 | by definition |

In [ ]:
SUN = dict(teff=5772, logg=4.438, feh=0.0)

print("LD coefficients for the Sun (IAU 2015)\n")

# kostogryz2022 — power-2 for TESS (no xi needed)
r = meidem.get_ld_coefficients(**SUN, passband='TESS',
                                grid='kostogryz2022', law='power2')
print(f"kostogryz2022  power-2    TESS   : {r['coefficients']}")

# kostogryz2022 — nonlinear (4-coeff) for TESS
r = meidem.get_ld_coefficients(**SUN, passband='TESS',
                                grid='kostogryz2022', law='nonlinear')
print(f"kostogryz2022  nonlinear  TESS   : {r['coefficients']}")

# claret2022 — power-2 for TESS
# Sun is a FGK dwarf → xi=2.0 km/s (Valenti & Fischer 2005)
r = meidem.get_ld_coefficients(**SUN, passband='TESS',
                                grid='claret2022', xi=2.0)
print(f"claret2022     power-2    TESS   : {r['coefficients']}")

# claret2017 — quadratic for TESS, ATLAS, Least-Squares
r = meidem.get_ld_coefficients(**SUN, passband='TESS',
                                grid='claret2017', law='quadratic',
                                mod='A', met='L', xi=2.0)
print(f"claret2017     quadratic  TESS   : {r['coefficients']}")

# claret2011 — quadratic for Kepler
r = meidem.get_ld_coefficients(**SUN, passband='Kp',
                                grid='claret2011', law='quadratic', xi=2.0)
print(f"claret2011     quadratic  Kepler : {r['coefficients']}")

---

## 3. Choosing the right grid

Each grid covers different passbands, atmospheric models, and LD laws. Use the table below as a quick reference:

| Situation | Recommended grid |
|-----------|------------------|
| TESS photometry, modern analysis | `kostogryz2022` |
| Kepler photometry | `claret2011` or `kostogryz2022` |
| Gaia, SDSS, Johnson, 2MASS passbands | `claret2022` |
| Cool stars (Teff < 3500 K) with TESS | `claret2017` with `mod='P'` |
| Comparing grids for a paper | `kostogryz2022` + `claret2022` |

### Power-2 vs quadratic law

The **power-2 law** shows the lowest residuals and smallest bias in recovered Rp/R* for TESS and Kepler (Maxted 2023). The **quadratic law** is the most widely used because it is supported by most transit fitting codes (batman, juliet, etc.).

In [ ]:
# Example: same star, different grids and laws for TESS
star = dict(teff=5500, logg=4.5, feh=-0.2)

r1 = meidem.get_ld_coefficients(**star, passband='TESS',
                                  grid='kostogryz2022', law='power2')

r2 = meidem.get_ld_coefficients(**star, passband='TESS',
                                  grid='claret2022', xi=2.0)

r3 = meidem.get_ld_coefficients(**star, passband='TESS',
                                  grid='claret2017', law='quadratic',
                                  mod='A', met='L', xi=2.0)

print(f"kostogryz2022  power-2    : {r1['coefficients']}")
print(f"claret2022     power-2    : {r2['coefficients']}")
print(f"claret2017     quadratic  : {r3['coefficients']}")

In [ ]:
# Example: cool M dwarf — use PHOENIX model (claret2017 mod='P')
# Note: kostogryz2022 does not cover Teff < 3500 K
cool_star = dict(teff=3500, logg=4.8, feh=0.0)

r = meidem.get_ld_coefficients(**cool_star, passband='TESS',
                                 grid='claret2017', law='quadratic', mod='P')

print(f"claret2017  PHOENIX  quadratic  TESS : {r['coefficients']}")
print(f"xi returned : {r['xi']}  (not applicable for PHOENIX)")

In [ ]:
# Example: Gaia passband — only claret2022 covers it
r = meidem.get_ld_coefficients(**SUN, passband='Gaia_G',
                                 grid='claret2022', xi=2.0)
print(f"claret2022  power-2  Gaia_G : {r['coefficients']}")

r = meidem.get_ld_coefficients(**SUN, passband='Gaia_BP',
                                 grid='claret2022', xi=2.0)
print(f"claret2022  power-2  Gaia_BP: {r['coefficients']}")

r = meidem.get_ld_coefficients(**SUN, passband='Gaia_RP',
                                 grid='claret2022', xi=2.0)
print(f"claret2022  power-2  Gaia_RP: {r['coefficients']}")

---

## 4. Choosing microturbulence (xi)

The `xi` parameter applies only to **ATLAS-based grids** (`claret2022`, `claret2017` with `mod='A'`, `claret2011` with `mod='A'`). It has no effect on `kostogryz2022` (MPS-ATLAS) or PHOENIX models.

**Always set `xi` explicitly.** If omitted, MEIDEM defaults to 2.0 km/s and emits a `UserWarning`. Available values: **0, 1, 2, 4, 8** km/s.

| Stellar type | logg range | Recommended xi | Reference |
|---|---|---|---|
| FGK main-sequence dwarfs | 4.0 – 5.0 | 2.0 km/s | Valenti & Fischer (2005) |
| Subgiants / metal-poor | 3.5 – 4.0 | 1.0 km/s | Bruntt et al. (2010) |
| Giants | 2.0 – 3.5 | 4.0 km/s | Bruntt et al. (2010) |
| Supergiants | < 2.0 | 8.0 km/s | Gray (2008) |

In [ ]:
import warnings

# This will emit a UserWarning — xi not set explicitly
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    r = meidem.get_ld_coefficients(**SUN, passband='TESS', grid='claret2022')
    if w:
        print(f"Warning: {w[0].message}")

In [ ]:
# Effect of xi on the coefficients for a giant star (logg ~ 2.5)
giant = dict(teff=5000, logg=2.5, feh=0.0)

print("Effect of xi on LD coefficients for a giant star (Teff=5000, logg=2.5):\n")
for xi_val in [1, 2, 4]:
    r = meidem.get_ld_coefficients(**giant, passband='TESS',
                                    grid='claret2017', law='quadratic',
                                    mod='A', met='L', xi=xi_val)
    print(f"  xi={xi_val} km/s : {r['coefficients']}")

---

## 5. Processing a table of stars

A common use case is computing LD coefficients for a large sample of stars from a CSV file. The example below shows how to loop over a pandas DataFrame, handle out-of-grid errors gracefully, and store results.

In [ ]:
import pandas as pd
import numpy as np

# Create a synthetic sample of stars for demonstration
# In practice, replace this with pd.read_csv('your_stars.csv')
# Required columns: hostname, Teff, logg, FeH
sample = pd.DataFrame({
    'hostname': ['star_A', 'star_B', 'star_C', 'star_D', 'star_E'],
    'Teff'    : [5778,     4500,     6200,     3200,     7000],
    'logg'    : [4.44,     4.70,     4.10,     4.90,     4.20],
    'FeH'     : [0.00,    -0.30,     0.15,     0.00,     0.10],
})

print(sample)

In [ ]:
# Choose xi based on logg — following Bruntt et al. (2010) and Valenti & Fischer (2005)
def choose_xi(logg):
    if logg >= 4.0:
        return 2.0   # FGK main-sequence dwarfs
    elif logg >= 3.5:
        return 1.0   # subgiants
    elif logg >= 2.0:
        return 4.0   # giants
    else:
        return 8.0   # supergiants

results = []

for _, row in sample.iterrows():
    xi = choose_xi(row['logg'])
    try:
        r = meidem.get_ld_coefficients(
            teff     = row['Teff'],
            logg     = row['logg'],
            feh      = row['FeH'],
            passband = 'TESS',
            grid     = 'kostogryz2022',
            law      = 'power2',
        )
        results.append({
            'star'  : row['hostname'],
            'LD_c'  : r['coefficients'][0],
            'LD_alpha': r['coefficients'][1],
            'grid'  : r['grid'],
            'xi'    : xi,
            'status': 'OK',
        })
    except ValueError as e:
        # Star is outside the grid — log it and continue
        results.append({
            'star'  : row['hostname'],
            'LD_c'  : None,
            'LD_alpha': None,
            'grid'  : 'kostogryz2022',
            'xi'    : xi,
            'status': 'OUT_OF_GRID',
        })
        print(f"  [{row['hostname']}] OUT_OF_GRID: {e}")

df_ld = pd.DataFrame(results)
print()
print(df_ld)

In [ ]:
# For OUT_OF_GRID stars, try a fallback grid
# claret2017 with PHOENIX covers cool stars (Teff < 3500 K)

results_final = []

for _, row in sample.iterrows():
    xi = choose_xi(row['logg'])

    # First attempt: kostogryz2022
    try:
        r = meidem.get_ld_coefficients(
            teff=row['Teff'], logg=row['logg'], feh=row['FeH'],
            passband='TESS', grid='kostogryz2022', law='power2',
        )
        results_final.append({
            'star': row['hostname'],
            'LD_c': r['coefficients'][0], 'LD_alpha': r['coefficients'][1],
            'grid': 'kostogryz2022', 'law': 'power2', 'status': 'OK',
        })
        continue
    except ValueError:
        pass

    # Fallback: claret2017 PHOENIX for cool stars (feh=0.0 only)
    try:
        r = meidem.get_ld_coefficients(
            teff=row['Teff'], logg=row['logg'], feh=0.0,
            passband='TESS', grid='claret2017', law='quadratic', mod='P',
        )
        results_final.append({
            'star': row['hostname'],
            'LD_c': r['coefficients'][0], 'LD_alpha': r['coefficients'][1],
            'grid': 'claret2017/PHOENIX', 'law': 'quadratic', 'status': 'FALLBACK',
        })
        continue
    except ValueError as e:
        results_final.append({
            'star': row['hostname'],
            'LD_c': None, 'LD_alpha': None,
            'grid': None, 'law': None, 'status': 'FAILED',
        })

df_final = pd.DataFrame(results_final)
print(df_final)

---

## Summary

| Task | Recommended approach |
|------|---------------------|
| TESS transit fitting | `grid='kostogryz2022'`, `law='power2'` |
| Kepler transit fitting | `grid='claret2011'`, `law='quadratic'`, `xi` explicit |
| Gaia / SDSS / Johnson passbands | `grid='claret2022'`, `xi` explicit |
| Cool M dwarfs (Teff < 3500 K) | `grid='claret2017'`, `mod='P'` |
| Large stellar sample | Loop with `try/except ValueError` + fallback grid |
| xi selection | Use `choose_xi(logg)` function above |

**References:**
- Kostogryz et al. (2022), A&A — doi:[10.1051/0004-6361/202140376](https://doi.org/10.1051/0004-6361/202140376)
- Claret & Southworth (2022), A&A 664, A128 — doi:[10.1051/0004-6361/202244278](https://doi.org/10.1051/0004-6361/202244278)
- Claret (2017), A&A 600, A30 — doi:[10.1051/0004-6361/201629705](https://doi.org/10.1051/0004-6361/201629705)
- Claret & Bloemen (2011), A&A 529, A75 — doi:[10.1051/0004-6361/201116451](https://doi.org/10.1051/0004-6361/201116451)
- Valenti & Fischer (2005), ApJS 159, 141
- Bruntt et al. (2010), MNRAS 405, 1907
- Gray (2008), *The Observation and Analysis of Stellar Photospheres*, 3rd ed.